[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/sec-reference-data-python.ipynb)

# SEC Reference Data with Python -- Tickers, Exchanges, CIK Lookups

Use **edgartools** to access SEC reference data -- ticker-to-CIK mappings, exchange listings, industry codes, CUSIP lookups, form descriptions, and more. All free, no API key required.

**What you'll learn:**
- How ticker-to-CIK resolution works (and why it works offline)
- Browse companies by exchange, industry, and state
- Look up form descriptions and CUSIP-to-ticker mappings
- Create research datasets with company subsets
- Cache reference data locally for full offline use

## Install edgartools

In [ ]:
!pip install -U edgartools

## Setup

The SEC requires all automated tools to identify themselves. Replace the email below with your own -- any valid email works.

In [ ]:
import pandas as pd
from edgar import *

set_identity("your.name@example.com")

## Bundled Ticker Data -- Works Offline

edgartools ships with a bundled `company_tickers.parquet` file containing ~10,600 exchange-listed tickers. This means `Company()` lookups work **without an internet connection** for established tickers -- no setup required.

When you call `Company("AAPL")`, edgartools resolves the ticker using a three-level waterfall:

1. **Bundled parquet** (instant, no network) -- ~10,600 exchange-listed tickers
2. **Local downloaded data** (if configured) -- full SEC ticker universe
3. **Live SEC API** (fallback) -- for brand-new tickers like recent IPOs

Let's see what's in the bundled data:

In [ ]:
from edgar.reference.tickers import load_company_tickers_from_package

# Load the bundled ticker data (this is what Company() uses internally)
bundled = load_company_tickers_from_package()

print(f"Bundled tickers: {len(bundled):,}")
print(f"Columns: {list(bundled.columns)}")
print(f"Exchanges: {sorted(bundled['exchange'].dropna().unique())}")
print()
bundled.sample(10, random_state=42)

## Ticker-to-CIK Lookup

The most common use of reference data is resolving a ticker symbol to an SEC CIK number. This happens automatically when you use `Company()`:

In [ ]:
# Ticker lookup -- resolves ticker -> CIK -> full company data
company = Company("MSFT")
print(f"{company.name}")
print(f"  CIK:      {company.cik}")
print(f"  Ticker:   {company.ticker}")
print(f"  Exchange: {company.exchange}")
print(f"  Industry: {company.industry}")
print()

# You can also look up CIK directly without loading the full company
from edgar.reference.tickers import find_cik

cik = find_cik("NVDA")
print(f"NVDA CIK: {cik}")

## Browse Companies by Exchange

Get all companies listed on a specific stock exchange:

In [ ]:
from edgar.reference import get_companies_by_exchanges

# Get all NYSE-listed companies
nyse = get_companies_by_exchanges("NYSE")
print(f"NYSE companies: {len(nyse):,}")

# Get Nasdaq companies
nasdaq = get_companies_by_exchanges("Nasdaq")
print(f"Nasdaq companies: {len(nasdaq):,}")

# Multiple exchanges at once
major = get_companies_by_exchanges(["NYSE", "Nasdaq"])
print(f"NYSE + Nasdaq: {len(major):,}")
print()
major.head(10)

## Browse Companies by Industry

The SEC classifies companies using SIC (Standard Industrial Classification) codes. Use these to find companies in specific industries:

In [ ]:
from edgar.reference import get_companies_by_industry

# Software companies (SIC 7372)
software = get_companies_by_industry(sic=7372)
print(f"Software companies (SIC 7372): {len(software)}")
print()
software.head(10)

In [ ]:
# Pharmaceutical companies (SIC 2834)
pharma = get_companies_by_industry(sic=2834)
print(f"Pharmaceutical companies (SIC 2834): {len(pharma)}")
print()
pharma.head(10)

## Browse Companies by State

Find companies incorporated in a specific US state:

In [ ]:
from edgar.reference import get_companies_by_state

# Delaware -- the incorporation capital
delaware = get_companies_by_state("DE")
print(f"Delaware-incorporated companies: {len(delaware):,}")

# California
california = get_companies_by_state("CA")
print(f"California-incorporated companies: {len(california):,}")

# Texas
texas = get_companies_by_state("TX")
print(f"Texas-incorporated companies: {len(texas):,}")

## Popular Company Lists

edgartools includes curated lists of popular companies, useful for demos, testing, and quick analysis:

In [ ]:
from edgar.reference import (
    get_popular_companies,
    get_faang_companies,
    get_tech_giants,
    get_dow_jones_sample,
    PopularityTier,
)

# FAANG companies
faang = get_faang_companies()
print("FAANG:")
for _, row in faang.iterrows():
    print(f"  {row['ticker']:6s} {row['name']}")

print(f"\nTech Giants: {len(get_tech_giants())} companies")
print(f"Dow Jones sample: {len(get_dow_jones_sample())} companies")

# Popularity tiers
mega = get_popular_companies(PopularityTier.MEGA_CAP)
print(f"\nMega-cap ({len(mega)}): {', '.join(mega['ticker'].tolist())}")

## SEC Form Descriptions

Look up what any SEC form type means:

In [ ]:
from edgar.reference import describe_form

forms = ["10-K", "10-Q", "8-K", "S-1", "DEF 14A", "13F-HR", "SC 13D", "4"]

for form in forms:
    print(describe_form(form))

## CUSIP-to-Ticker Mapping

edgartools includes a CUSIP-to-ticker mapping, primarily used by 13F institutional holdings parsers. You can also use it directly:

In [ ]:
from edgar.reference import cusip_ticker_mapping, get_ticker_from_cusip

# Look up a single CUSIP
ticker = get_ticker_from_cusip("037833100")  # Apple's CUSIP
print(f"CUSIP 037833100 -> {ticker}")

# See the full mapping
mapping = cusip_ticker_mapping()
print(f"\nTotal CUSIP mappings: {len(mapping):,}")
print(f"Columns: {list(mapping.columns)}")
mapping.head(10)

## Place Codes and Filer Types

The SEC uses codes for states and countries. edgartools can decode these and classify filers:

In [ ]:
from edgar.reference import (
    get_place_name,
    get_filer_type,
    is_us_company,
    is_foreign_company,
    is_canadian_company,
)

# Decode SEC place codes
codes = ["DE", "CA", "NY", "X0", "L2", "A6"]
for code in codes:
    name = get_place_name(code)
    ftype = get_filer_type(code)
    print(f"  {code:4s} -> {name or 'Unknown':30s} ({ftype or 'Unknown'})")

print()
print(f"Is 'DE' a US company?      {is_us_company('DE')}")
print(f"Is 'X0' a foreign company?  {is_foreign_company('X0')}")
print(f"Is 'A6' Canadian?           {is_canadian_company('A6')}")

## Build Research Datasets with CompanySubset

The `CompanySubset` fluent interface lets you chain operations to build precise company selections:

In [ ]:
from edgar.reference import CompanySubset

# 50 random NYSE/Nasdaq companies (reproducible)
research_set = (CompanySubset()
    .from_exchange(["NYSE", "Nasdaq"])
    .sample(50, random_state=42)
    .get())

print(f"Research set: {len(research_set)} companies")
print(f"Exchange breakdown:")
print(research_set["exchange"].value_counts().to_string())
print()
research_set.head(10)

In [ ]:
from edgar.reference import get_stratified_sample

# Stratified sample -- maintains exchange proportions
stratified = get_stratified_sample(n=100, stratify_by="exchange", random_state=42)

print(f"Stratified sample: {len(stratified)} companies")
print(f"\nExchange proportions:")
print(stratified["exchange"].value_counts(normalize=True).round(3).to_string())

## Industry-Specific Convenience Functions

Quickly get companies in common industries:

In [ ]:
from edgar.reference import (
    get_banking_companies,
    get_pharmaceutical_companies,
    get_software_companies,
    get_semiconductor_companies,
    get_oil_gas_companies,
    get_real_estate_companies,
)

industries = {
    "Banking": get_banking_companies,
    "Pharma": get_pharmaceutical_companies,
    "Software": get_software_companies,
    "Semiconductors": get_semiconductor_companies,
    "Oil & Gas": get_oil_gas_companies,
    "Real Estate": get_real_estate_companies,
}

print(f"{'Industry':<20s} {'Companies':>10s}")
print(f"{'-'*20} {'-'*10}")
for name, func in industries.items():
    companies = func()
    print(f"{name:<20s} {len(companies):>10,}")

## Full Offline Setup

The bundled parquet covers ~10,600 exchange-listed tickers. For the full SEC ticker universe (including recent IPOs, mutual funds, and non-exchange entities), download reference data locally:

```python
from edgar import download_edgar_data, use_local_storage

# One-time download of reference data (~50 MB, takes a few seconds)
download_edgar_data(submissions=False, facts=False, reference=True)

# Enable local storage -- now all lookups prefer local data
use_local_storage()

# Works fully offline, even for recent IPOs
company = Company("AAPL")
```

The downloaded reference data includes:

| File | Content |
|------|---------|
| `company_tickers.json` | All SEC-registered tickers with CIK |
| `company_tickers_exchange.json` | Tickers with exchange information |
| `mutual_fund_tickers.json` | Mutual fund ticker-to-CIK mappings |

For more on local storage, see the [Local Storage guide](https://edgartools.readthedocs.io/guides/local-storage/).

## Quick Reference

```python
from edgar import *
from edgar.reference import *
from edgar.reference.tickers import find_cik

set_identity("your.name@example.com")

# ── Ticker / CIK resolution ──
company = Company("AAPL")               # Ticker -> full company object
cik = find_cik("AAPL")                   # Ticker -> CIK only (lightweight)

# ── Companies by exchange ──
get_companies_by_exchanges("NYSE")       # All NYSE companies
get_companies_by_exchanges(["NYSE", "Nasdaq"])  # Multiple exchanges

# ── Companies by industry / state ──
get_companies_by_industry(sic=7372)      # By SIC code
get_companies_by_state("DE")             # By state of incorporation

# ── Popular companies ──
get_popular_companies()                  # All popular companies
get_faang_companies()                    # FAANG
get_tech_giants()                        # Major tech companies

# ── Industry convenience ──
get_banking_companies()                  # Banks
get_pharmaceutical_companies()           # Pharma
get_software_companies()                 # Software

# ── Form descriptions ──
describe_form("10-K")                    # "Form 10-K: Annual report"

# ── CUSIP lookups ──
get_ticker_from_cusip("037833100")       # CUSIP -> ticker
cusip_ticker_mapping()                   # Full CUSIP DataFrame

# ── Place codes ──
get_place_name("DE")                     # "Delaware"
get_filer_type("DE")                     # "Domestic"
is_us_company("DE")                      # True

# ── Research datasets ──
CompanySubset().from_exchange("NYSE").sample(100, random_state=42).get()
get_stratified_sample(n=100, stratify_by="exchange")
get_random_sample(n=50)

# ── Offline setup ──
download_edgar_data(submissions=False, facts=False, reference=True)
use_local_storage()
```

## What's Next

You've learned how to access and use SEC reference data. Here are related tutorials:

- [SEC Company Data](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/sec-company-data-python.ipynb) -- Company metadata, filing history, financials
- [Search and Filter SEC Filings](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/search-sec-filings-python.ipynb) -- Find specific filings
- [SEC Industry SIC Codes](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/sec-industry-sic-code-python.ipynb) -- Industry classification deep dive
- [Ticker Search](https://colab.research.google.com/github/dgunning/edgartools/blob/main/notebooks/Ticker-Search-with-edgartools.ipynb) -- Advanced ticker search techniques

**Resources:**
- [EdgarTools Documentation](https://edgartools.readthedocs.io/)
- [GitHub Repository](https://github.com/dgunning/edgartools)
- [PyPI Package](https://pypi.org/project/edgartools/)

---

## Support EdgarTools

If you found this tutorial helpful, here are a few ways to support the project:

- **Star the repo** -- [github.com/dgunning/edgartools](https://github.com/dgunning/edgartools) -- it helps others discover edgartools
- **Visit edgartools.io** -- [edgartools.io](https://www.edgartools.io/) -- for more tutorials, articles, and updates
- **Report issues** -- found a bug or have a feature idea? [Open an issue](https://github.com/dgunning/edgartools/issues)
- **Share this notebook** -- know someone who works with SEC data? Send them the Colab link

*edgartools is free, open-source, and community-driven. No API key or paid subscription required.*